# Quantization

Reduce model weight precision from FP32 to INT8 — up to 4× smaller on disk, faster CPU inference, minimal accuracy loss.

Demo covers two PyTorch approaches on the same MNIST MLP baseline:
- **Dynamic quantization** — weights to INT8 at conversion time, no calibration needed
- **Static PTQ** — weights + activations to INT8, requires a small calibration pass

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import random, numpy as np, os, time

In [ ]:
# ----------------------------
# Setup
# ----------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", DEVICE)

## Data

In [ ]:
# ----------------------------
# Data (MNIST)
# ----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
calib_loader = DataLoader(train_ds, batch_size=64,  shuffle=True)   # small subset for PTQ calibration

## FP32 Baseline

Standard 32-bit floating-point model. All quantized variants start from these weights.

In [ ]:
# ----------------------------
# Model
# ----------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
# ----------------------------
# Train / Eval helpers
# ----------------------------
def train_one_epoch(model, loader, opt, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        opt.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total   += x.size(0)
    return correct / total, total_loss / total

@torch.no_grad()
def evaluate(model, loader, device="cpu"):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / total

def file_kb(path):
    return os.path.getsize(path) / 1024 if os.path.exists(path) else 0.0

In [ ]:
# ----------------------------
# Train + save FP32
# ----------------------------
model = MLP().to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)

for ep in range(3):
    tr_acc, tr_loss = train_one_epoch(model, train_loader, opt, DEVICE)
    te_acc          = evaluate(model, test_loader, DEVICE)
    print(f"Epoch {ep+1}/3 | loss {tr_loss:.4f} | train acc {tr_acc:.4f} | test acc {te_acc:.2f}%")

torch.save(model.state_dict(), "fp32.pth")
fp32_acc  = evaluate(model, test_loader, DEVICE)
fp32_size = file_kb("fp32.pth")
print(f"\nFP32 baseline — size: {fp32_size:.1f} KB | acc: {fp32_acc:.2f}%")

## Dynamic Quantization

Weights are quantized to INT8 at conversion time. Activations are quantized on-the-fly during inference.  
No calibration data needed — the easiest quantization to apply.

In [ ]:
# ----------------------------
# Dynamic quantization
# Quantization runs on CPU only — move model off GPU first
# ----------------------------
model_cpu = MLP()
model_cpu.load_state_dict(torch.load("fp32.pth", map_location="cpu", weights_only=True))
model_cpu.eval()

model_dynamic = torch.quantization.quantize_dynamic(
    model_cpu, {nn.Linear}, dtype=torch.qint8
)

torch.save(model_dynamic.state_dict(), "dynamic_int8.pth")
dyn_acc  = evaluate(model_dynamic, test_loader, "cpu")
dyn_size = file_kb("dynamic_int8.pth")
print(f"Dynamic INT8 — size: {dyn_size:.1f} KB | acc: {dyn_acc:.2f}%")
print(f"Size vs FP32: {fp32_size / dyn_size:.2f}x")

## Static Post-Training Quantization (PTQ)

Both weights **and** activations are quantized to INT8.  
Requires a calibration pass to record activation min/max ranges before conversion.

The model needs `QuantStub` / `DeQuantStub` wrappers to mark quantization boundaries.

In [ ]:
# ----------------------------
# Quantizable model (same weights, PTQ wrappers added)
# ----------------------------
class QuantizableMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant   = torch.quantization.QuantStub()
        self.fc1     = nn.Linear(28*28, 256)
        self.fc2     = nn.Linear(256, 128)
        self.fc3     = nn.Linear(128, 10)
        self.relu    = nn.ReLU()
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.quant(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return self.dequant(x)

In [ ]:
# ----------------------------
# Calibrate + convert to INT8
# ----------------------------
model_static = QuantizableMLP()
model_static.load_state_dict(
    torch.load("fp32.pth", map_location="cpu", weights_only=True),
    strict=False   # quant/dequant stubs have no pretrained weights
)
model_static.eval()

model_static.qconfig = torch.quantization.get_default_qconfig("fbgemm")
torch.quantization.prepare(model_static, inplace=True)

# calibration — collect activation ranges over ~640 samples
with torch.no_grad():
    for i, (x, _) in enumerate(calib_loader):
        model_static(x)
        if i >= 10:
            break

torch.quantization.convert(model_static, inplace=True)

torch.save(model_static.state_dict(), "int8.pth")
static_acc  = evaluate(model_static, test_loader, "cpu")
static_size = file_kb("int8.pth")
print(f"Static INT8  — size: {static_size:.1f} KB | acc: {static_acc:.2f}%")
print(f"Size vs FP32: {fp32_size / static_size:.2f}x")

## Results

Compare file size, accuracy, and CPU inference latency across all three variants.

In [ ]:
# ----------------------------
# Latency comparison (CPU)
# ----------------------------
def latency_ms(model, loader, device="cpu", n_batches=20):
    model.eval()
    t0 = time.perf_counter()
    with torch.no_grad():
        for i, (x, _) in enumerate(loader):
            model(x.to(device))
            if i >= n_batches:
                break
    return (time.perf_counter() - t0) / (n_batches + 1) * 1000

model.to("cpu")   # move FP32 to CPU for fair comparison

fp32_lat   = latency_ms(model,          test_loader)
dyn_lat    = latency_ms(model_dynamic,   test_loader)
static_lat = latency_ms(model_static,    test_loader)

print(f"{'':25} {'Size (KB)':>10} {'Acc (%)':>10} {'Latency (ms/batch)':>20}")
print("-" * 68)
print(f"{'FP32 baseline':25} {fp32_size:>10.1f} {fp32_acc:>10.2f} {fp32_lat:>20.1f}")
print(f"{'Dynamic INT8':25} {dyn_size:>10.1f} {dyn_acc:>10.2f} {dyn_lat:>20.1f}")
print(f"{'Static INT8 (PTQ)':25} {static_size:>10.1f} {static_acc:>10.2f} {static_lat:>20.1f}")